In [ ]:
import pandas as pd
import numpy as np

#Cargamos el dataset ya tratado y hacemos merge para unir las etiquetas de cada posst
df = pd.read_csv("top50_posts_per_user_reduced.csv")       # autor, trait, post, similarity
labels = pd.read_csv("authors_train.csv")  # username + 5 rasgos

df = df.merge(labels, on="username")

df.head()




,username,trait,post,similarity,agreeableness,openness,conscientiousness,extraversion,neuroticism
0,-Areopagan-,Agreeableness,I am smarter than you and will work you into d...,0.476548,0.0,99.0,96.0,60.0,1.0
1,-Areopagan-,Agreeableness,"I have two friends. I alienate everyone, event...",0.472431,0.0,99.0,96.0,60.0,1.0
2,-Areopagan-,Agreeableness,Yeah I wouldnt want to deal with someone like ...,0.390348,0.0,99.0,96.0,60.0,1.0
3,-Areopagan-,Agreeableness,your first and second question is the same que...,0.328232,0.0,99.0,96.0,60.0,1.0
4,-Areopagan-,Openness,I am smarter than you and will work you into d...,0.357194,0.0,99.0,96.0,60.0,1.0


In [ ]:
#esta funcion lo que busca es indicar cual es la columna objetivo en cada fila, en base al rasgo de la columna trait. Coge como etiqueta para ese post la columna (de entre las 5 etiquetas) que se llame igual al valor que hay en trait
#por ejemplo en la primera fila la columna trait tiene agreeablenes, entonces, para ese post la etiqueta que se busca es Agreeableness

def get_target(row):
    return str(row[row["trait"].lower()])  # valor numérico correcto

df["input_text"] = df.apply(lambda row: f"predict this trait: {row['trait']}: {row['post']}", axis=1)  #algo asi : "predict this trait: agreeableness: Today I felt very happy helping my friends"

df["target_text"] = df.apply(get_target, axis=1) #creamos la columna que contiene la etiqueta buscada para cada post

In [ ]:
from datasets import Dataset

# Creamos un dataset de HuggingFace a partir del DataFrame, conservando únicamente las columnas que T5 usará:
#input_text -> instrucción + post
#target_text -> valor del rasgo a predecir.

dataset = Dataset.from_pandas(df[["input_text", "target_text"]]) 
dataset = dataset.train_test_split(test_size=0.1)

In [ ]:
from transformers import T5Tokenizer

# Cargamos el tokenizer de T5-small. Este objeto transforma texto en IDs de tokens y también se utiliza para decodificar las predicciones del modelo.
tokenizer = T5Tokenizer.from_pretrained("t5-small")

#Tokenizamos el texto de entrada y tambien el objetivo (input text y target text)
def preprocess(example):
    enc = tokenizer(
        example["input_text"],
        padding="max_length", #rellenamos hasta la longitud maxima
        truncation=True, #esto sirve para cortar, en caso de que excedamos longitud con el post (256)
        max_length=256,
    )
    with tokenizer.as_target_tokenizer(): #lo mismo ahora para target text
        labels = tokenizer(
            example["target_text"],
            padding="max_length",
            truncation=True,
            max_length=10 #es pequeño porque el objetivo es solo una palabra
        )
    
     # Añadimos los input_ids del target como "labels" que es el formato esperado por el Trainer de HuggingFace.
    enc["labels"] = labels["input_ids"]
    return enc

# Aplicamos el preprocesado a todo el dataset y "batched=True" permite procesar varias muestras en paralelo (más eficiente).
tokenized = dataset.map(preprocess, batched=True)


Map:   0%|          | 0/139886 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/15543 [00:00<?, ? examples/s]

In [ ]:
from transformers import T5ForConditionalGeneration, TrainingArguments, Trainer

model = T5ForConditionalGeneration.from_pretrained("t5-small") #cargamos el modelo que vamos a usar, en este caso el pequeño que ya tarda de por sí unas 4/5 horas en entrenar nuestro dataset


In [ ]:
#Configuracion del entrenamiento
training_args = TrainingArguments(
    output_dir="./t5_personality",
    per_device_train_batch_size=8, #Tamaño de batch para entrenamiento
    per_device_eval_batch_size=8, #Tamaño del batch para eval
    report_to="none", #Desactiva integración con wandb/tensorboard (da problemas)
    learning_rate=3e-4,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=100,  # Frecuencia con la que se mostrarán logs del entrenamiento
    save_total_limit=2, # Solo se guardarán los últimos 2 checkpoints
)

In [ ]:
#Juntamos todo en el trainer listo para lanazar
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
)


In [ ]:
trainer.train()


Step,Training Loss
100,1.427400
200,0.522200
300,0.501400
400,0.493200
500,0.489900
600,0.482400
700,0.475800
800,0.480500
900,0.475200
1000,0.469500


In [ ]:
#Función para predecir un post con T5
def predict_trait_post(model, tokenizer, trait, post):
    input_text = f"predict this trait {trait}: {post}" #Instrucción otra vez en lenguaje natural que T5 debe procesar.
    #Tokenizamos la entrada y la movemos al dispositivo del modelo
    inputs = tokenizer(
        input_text,
        return_tensors="pt", #se devuelven tensores pytorch
        truncation=True,
        padding=True
    ).to(model.device)

    outputs = model.generate(**inputs, max_length=10)     # Generamos una predicción textual. max_length=5 es suficiente porque la salida es numero corto.

    pred = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()   # Decodificamos la secuencia de tokens generada para obtener el número como string.

    #sIntentamos convertir la predicción a float. Si T5 genera algo no numérico, devolvemos NaN.
    try:
        return float(pred)
    except:
        return np.nan



In [ ]:
# Calculamos la predicción del modelo para cada fila del DataFrame.
# Para cada post y su rasgo asociado (trait), aplicamos la función
# predict_trait_post(), que genera un valor textual con T5 y lo convierte en float.

df["predicted"] = df.apply(
    lambda row: predict_trait_post(
        model, tokenizer, row["trait"], row["post"]
    ),
    axis=1
)


In [ ]:
#Comparar el numero generado por t5 con el real
df["target_value"] = df.apply(
    lambda row: row[row["trait"].lower()],
    axis=1
)


In [ ]:
#Cálculo del RMSE por cada rasgo de personalidad.

from sklearn.metrics import mean_squared_error

traits = ["openness", "conscientiousness", "extraversion",
          "agreeableness", "neuroticism"]

rmse_by_trait = {}

for t in traits:
    subset = df[df["trait"].str.lower() == t] #seleccionamos todas las filas del trait t 
    # Calculamos el RMSE entre el valor real del rasgo (target_value)
    # y la predicción generada por T5 (predicted).
    rmse = mean_squared_error(subset["target_value"], subset["predicted"], squared=False)
    rmse_by_trait[t] = rmse

rmse_by_trait
